In [1]:
import csv
import sys
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

WORKDIR = Path.cwd()
MEENT_ROOT = WORKDIR.parents[2]
sys.path.insert(0, str(MEENT_ROOT))

import meent

print("meent   :", meent.__file__)
print("workdir :", WORKDIR)
assert str(MEENT_ROOT) in meent.__file__, (
    "meent came from elsewhere - restart the kernel so the sys.path insert wins"
)

meent   : e:\Taesang Yun\meent\meent\__init__.py
workdir : e:\Taesang Yun\meent\validation\transmittance\1D_anisotropic_grating_conical_incidence


In [2]:
QUICK = False

n_top = 1.0
n_bot = 1.0
theta = 30 * np.pi / 180
phi = 30 * np.pi / 180          # conical mount: incidence out of the x-z plane
period = [1000e-9]
thickness = [500e-9, 500e-9]
type_complex = torch.complex128

# Diagonal anisotropy (nx, ny, nz), the same tensor for the grating bars and the
# slab.  RETICOLO gets the identical structure through parm.res1.change_index.
n_x, n_y, n_z = 1.5, 2.0, 2.5
n_air = [1.0, 1.0, 1.0]
n_grating = [n_x, n_y, n_z]
n_slab = [n_x, n_y, n_z]

grating_width = 500e-9
grating_center = 500e-9

fto = [100]

if QUICK:
    wavelength = np.arange(500, 701, 20) * 1e-9
else:
    wavelength = np.arange(500, 701, 1) * 1e-9

METHODS = ['discrete', 'continuous', 'vector']
POLS = {0: 'TE', 1: 'TM'}

print(f"{'QUICK' if QUICK else 'FULL'} run")
print(f"  wavelengths : {len(wavelength)}  "
      f"({wavelength[0]*1e9:.0f}-{wavelength[-1]*1e9:.0f} nm)")
print(f"  incidence   : conical (theta = {np.degrees(theta):.1f} deg, "
      f"phi = {np.degrees(phi):.1f} deg, "
      f"k_parallel = n_top*sin(theta) = {n_top*np.sin(theta):.6f})")
print(f"  index       : nx = {n_x}, ny = {n_y}, nz = {n_z}")
print(f"  fto         : {fto}")
print(f"  total solves: {len(METHODS) * len(POLS) * len(wavelength)}")

FULL run
  wavelengths : 201  (500-700 nm)
  incidence   : conical (theta = 30.0 deg, phi = 30.0 deg, k_parallel = n_top*sin(theta) = 0.500000)
  index       : nx = 1.5, ny = 2.0, nz = 2.5
  fto         : [100]
  total solves: 1206


In [3]:
def build_raster():
    # (Layers, H, W, 3) - the trailing 3 is what marks the ucell anisotropic.
    layer1 = [n_air, n_grating, n_grating, n_air]
    layer2 = [n_slab] * 4
    return np.array([[layer1], [layer2]], dtype=float)

def build_vector():
    return [
        [n_air, [['rectangle', grating_center, grating_center,
                  grating_width, period[0], n_grating]]],
        [n_slab, []],
    ]

def make_mee(method, pol, wl):
    kwargs = dict(backend=2, pol=pol, n_top=n_top, n_bot=n_bot, fto=fto,
                  wavelength=wl, period=period, thickness=thickness,
                  type_complex=type_complex, theta=theta, phi=phi)
    if method == 'vector':
        return meent.call_mee(ucell=build_vector(), **kwargs)
    ft = 0 if method == 'discrete' else 1
    return meent.call_mee(ucell=build_raster(), fourier_type=ft, **kwargs)

def solve_one(method, pol, wl):
    r = make_mee(method, pol, wl).conv_solve().res
    return float(r.de_ri.sum()), float(r.de_ti.sum())

def out_name(prefix, pol, method=None):
    tail = f'_{method}' if method else ''
    return f'{prefix}_1D_anisotropic_grating_conical_incidence_{POLS[pol]}{tail}.txt'

def save_rows(fname, rows):
    with open(WORKDIR / fname, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['lambda_nm', 'R', 'T'])
        w.writerows(rows)

def load_rows(fname):
    p = WORKDIR / fname
    if not p.exists():
        return None
    a = np.genfromtxt(p, delimiter=',', skip_header=1)
    return a.reshape(1, -1) if a.ndim == 1 else a

print('helpers ready')

# Which solver each method lands in.  Under conical incidence (phi != 0) a 1D
# grating couples the two polarizations, so an anisotropic ucell routes to the
# full 2D solver (grating type 2) for every method - discrete, continuous and
# vector alike, unlike the normal / oblique cases where the raster paths drop to
# the fast 1D solver.  pol selects the incident polarization through psi
# (pol=0 -> TE, pol=1 -> TM), and .res for those matches RETICOLO's TEinc /
# TMinc from a single conical solve.
for _m in METHODS:
    _mee = make_mee(_m, 0, wavelength[0])
    _shape = 'vector' if _m == 'vector' else tuple(_mee.ucell.shape)
    print(f'  {_m:11s} ucell={_shape}  is_aniso={_mee.is_aniso}  '
          f'grating_type={_mee.grating_type_assigned}')

helpers ready
  discrete    ucell=(2, 1, 4, 3)  is_aniso=True  grating_type=2
  continuous  ucell=(2, 1, 4, 3)  is_aniso=True  grating_type=2
  vector      ucell=vector  is_aniso=False  grating_type=2


In [4]:
t0 = time.time()
failures = []

for method in METHODS:
    for pol in POLS:
        rows = []
        for wl in wavelength:
            try:
                R, T = solve_one(method, pol, wl)
            except Exception as exc:
                R = T = np.nan
                failures.append((method, POLS[pol], wl, repr(exc)))
            rows.append([wl * 1e9, R, T])
        fname = out_name('Meent', pol, method)
        save_rows(fname, rows)
        arr = np.array(rows)
        ok = np.isfinite(arr[:, 1])
        print(f'  {method:11s} {POLS[pol]}  {ok.sum():4d}/{len(arr):4d} finite  ->  {fname}')

print(f'\nmeent sweep done in {time.time() - t0:.1f} s, {len(failures)} failed solves')
for f in failures[:5]:
    print('   ', f)
if len(failures) > 5:
    print(f'    ... and {len(failures) - 5} more')

  discrete    TE   201/ 201 finite  ->  Meent_1D_anisotropic_grating_conical_incidence_TE_discrete.txt
  discrete    TM   201/ 201 finite  ->  Meent_1D_anisotropic_grating_conical_incidence_TM_discrete.txt
  continuous  TE   201/ 201 finite  ->  Meent_1D_anisotropic_grating_conical_incidence_TE_continuous.txt
  continuous  TM   201/ 201 finite  ->  Meent_1D_anisotropic_grating_conical_incidence_TM_continuous.txt
  vector      TE   201/ 201 finite  ->  Meent_1D_anisotropic_grating_conical_incidence_TE_vector.txt
  vector      TM   201/ 201 finite  ->  Meent_1D_anisotropic_grating_conical_incidence_TM_vector.txt

meent sweep done in 587.1 s, 0 failed solves


In [ ]:
# Energy conservation, meent AND RETICOLO - and *where* it breaks.
#
# Every index in the structure is real and n_top = n_bot = 1, so nothing absorbs
# and nothing leaks into a substrate: R + T must be 1 at every wavelength.  That
# makes this an absolute check on each result file taken on its own - it never
# compares meent against RETICOLO, so a solver can fail here even when the two
# agree with each other, and a missing RETICOLO file is reported as missing
# rather than silently skipped.
#
# R and T are computed from independent field amplitudes (different amplitude
# vectors, different kz, different normalization), never as T = 1 - R, so this
# is a real check and not an identity.  It is necessary but not sufficient:
# R and T can both be wrong while their sum is 1, which is what the RETICOLO
# comparison further down is for.
#
# The summary table is the headline; the listing under it names every wavelength
# that broke the tolerance, collapsed into contiguous bands so that a mass
# failure stays readable instead of scrolling past as hundreds of lines.
TOL_ENERGY = 1e-9
N_SHOW = 25          # max bands listed per file

cases = [(f'meent {method}', out_name('Meent', pol, method), pol)
         for method in METHODS for pol in POLS]
cases += [('RETICOLO', out_name('RETICOLO', pol), pol) for pol in POLS]


def contiguous_bands(lams, step):
    """Split ascending wavelengths into runs of consecutive sweep points."""
    if len(lams) == 0:
        return []
    gaps = np.where(np.diff(lams) > 1.5 * step)[0]
    return np.split(np.arange(len(lams)), gaps + 1)


n_energy_fail = 0
n_missing = 0
n_checked = 0        # files that were actually present and read
offenders = []       # files that broke the tolerance
nan_rows = []        # (label, pol, lambdas) where the solve produced no number

print(f"{'source':16s} {'pol':4s} {'n':>4s} {'max|R+T-1|':>13s} "
      f"{'at lambda':>11s} {'mean(R+T-1)':>13s}")
print('-' * 68)
for label, fname, pol in cases:
    a = load_rows(fname)
    if a is None:
        print(f'{label:16s} {POLS[pol]:4s} {"-":>4s} {"file missing":>13s}')
        n_missing += 1
        continue

    n_checked += 1
    lam = a[:, 0]
    resid = a[:, 1] + a[:, 2] - 1
    finite = np.isfinite(resid)
    step = np.median(np.diff(lam)) if len(lam) > 1 else 1.0

    if (~finite).any():
        nan_rows.append((label, pol, lam[~finite]))

    if not finite.any():
        print(f'{label:16s} {POLS[pol]:4s} {0:4d} {"all NaN":>13s}')
        n_energy_fail += 1
        continue

    lam_f, resid_f = lam[finite], resid[finite]
    err = np.abs(resid_f)
    i = int(np.argmax(err))
    over = err > TOL_ENERGY

    if over.any():
        n_energy_fail += 1
        offenders.append((label, pol, lam_f[over], resid_f[over],
                          step, int(finite.sum())))

    # A signed mean well above the noise floor means a systematic leak; a signed
    # mean near zero with a large max means a few isolated bad wavelengths.
    mark = f'  <-- {over.sum()}/{finite.sum()} over {TOL_ENERGY:g}' if over.any() else ''
    print(f'{label:16s} {POLS[pol]:4s} {finite.sum():4d} {err[i]:13.3e} '
          f'{lam_f[i]:9.0f}nm {resid_f.mean():13.3e}{mark}')

# ---- which wavelengths, exactly
if offenders:
    print()
    print(f'=== wavelengths where |R + T - 1| > {TOL_ENERGY:g} ===')
    print('Consecutive wavelengths are collapsed into bands.  A wide band is a')
    print('systematic problem over a region; scattered single points are isolated')
    print('anomalies (a Rayleigh wavelength, a degenerate order).')
    for label, pol, lams, resids, step, n_total in offenders:
        order = np.argsort(lams)
        lams, resids = lams[order], resids[order]
        bands = contiguous_bands(lams, step)
        print(f'\n{label} {POLS[pol]}  -  {len(lams)}/{n_total} wavelengths '
              f'in {len(bands)} band(s)')
        for idx in bands[:N_SHOW]:
            b_lam, b_res = lams[idx], resids[idx]
            w = int(np.argmax(np.abs(b_res)))
            if len(idx) == 1:
                print(f'      {b_lam[0]:7.1f} nm{"":14s}'
                      f'R + T - 1 = {b_res[0]:+.3e}')
            else:
                print(f'      {b_lam[0]:7.1f} - {b_lam[-1]:7.1f} nm '
                      f'({len(idx):4d} pts)  worst {b_res[w]:+.3e} '
                      f'@ {b_lam[w]:.1f} nm')
        if len(bands) > N_SHOW:
            hidden = sum(len(idx) for idx in bands[N_SHOW:])
            print(f'      ... and {len(bands) - N_SHOW} more band(s), '
                  f'{hidden} wavelength(s) (raise N_SHOW to list them)')

if nan_rows:
    print()
    print('=== wavelengths with no finite result (failed solve) ===')
    for label, pol, lams in nan_rows:
        shown = ', '.join(f'{x:.0f}' for x in lams[:N_SHOW])
        more = f' ... (+{len(lams) - N_SHOW} more)' if len(lams) > N_SHOW else ''
        print(f'  {label} {POLS[pol]}  {len(lams)} point(s): {shown} nm{more}')

print()
if n_checked == 0:
    # Nothing was read, so there is no result to pass or fail: say so instead of
    # reporting a hollow PASS.  Run the meent sweep cell above first, and the .m
    # file for the RETICOLO columns; this cell only reads what they wrote.
    print('energy conservation: no result files found - nothing was checked')
    print('  run the meent sweep cell above (writes the Meent_*.txt files),')
    print('  and the .m file (writes the RETICOLO_*.txt files), then re-run this cell')
elif n_energy_fail:
    print(f'energy conservation: {n_energy_fail}/{n_checked} file(s) deviate from '
          f'R + T = 1 by more than {TOL_ENERGY:g}')
else:
    print(f'energy conservation: all {n_checked} file(s) within {TOL_ENERGY:g} of '
          f'R + T = 1  ->  PASS')
if n_missing:
    print(f'({n_missing} file(s) still missing - run the .m file, and the sweep above)')

In [ ]:
def compare(a, b):
    m = np.isfinite(a[:, 1]) & np.isfinite(b[:, 1])
    return np.abs(a[m, 1] - b[m, 1]), np.abs(a[m, 2] - b[m, 2])

print(f"{'pol':4s} {'method':11s} "
      f"{'max|dR|':>11s} {'med|dR|':>11s} {'max|dT|':>11s}   (reference: discrete)")
print('-' * 68)
for pol in POLS:
    ref = load_rows(out_name('Meent', pol, 'discrete'))
    if ref is None:
        continue
    for method in METHODS[1:]:
        a = load_rows(out_name('Meent', pol, method))
        if a is None:
            continue
        dR, dT = compare(ref, a)
        print(f'{POLS[pol]:4s} {method:11s} '
              f'{dR.max():11.3e} {np.median(dR):11.3e} {dT.max():11.3e}')

print()
print('--- continuous vs vector (should be ~machine precision) ---')
print('Both run the 2D solver here (conical), so agreement says the raster-')
print('continuous and vector convolution matrices land on the same result.')
for pol in POLS:
    a = load_rows(out_name('Meent', pol, 'continuous'))
    b = load_rows(out_name('Meent', pol, 'vector'))
    if a is None or b is None:
        continue
    dR, dT = compare(a, b)
    print(f'{POLS[pol]:4s} max|dR| {dR.max():.3e}  med|dR| {np.median(dR):.3e}  '
          f'max|dT| {dT.max():.3e}')

In [ ]:
TOL_RETICOLO = {'discrete': 1e-4, 'continuous': 1e-8, 'vector': 1e-6}

found_any = False
worst = []
n_checked = 0
n_bad = 0

print(f"{'pol':4s} {'method':11s} "
      f"{'max|dR|':>11s} {'med|dR|':>11s} {'max|dT|':>11s}  n")
print('-' * 60)
for pol in POLS:
    ret = load_rows(out_name('RETICOLO', pol))
    if ret is None:
        print(f'{POLS[pol]:4s}  RETICOLO file missing - run the .m file')
        continue
    found_any = True
    for method in METHODS:
        me = load_rows(out_name('Meent', pol, method))
        if me is None:
            continue
        dR, dT = [], []
        for row in me:
            j = np.where(np.isclose(ret[:, 0], row[0], atol=1e-6))[0]
            if j.size == 0:
                continue
            rr = ret[j[0]]
            if not (np.isfinite(rr[1]) and np.isfinite(row[1])):
                continue
            dR.append(abs(rr[1] - row[1]))
            dT.append(abs(rr[2] - row[2]))
            worst.append((dR[-1], pol, method, row[0]))
        if not dR:
            continue
        dR, dT = np.array(dR), np.array(dT)
        tol = TOL_RETICOLO[method]
        n_over = int(np.sum(np.maximum(dR, dT) > tol))
        n_checked += len(dR)
        n_bad += n_over
        mark = f'  <-- {n_over} over {tol:g}' if n_over else ''
        print(f'{POLS[pol]:4s} {method:11s} '
              f'{dR.max():11.3e} {np.median(dR):11.3e} {dT.max():11.3e} '
              f'{len(dR):3d}{mark}')

print()
if not found_any:
    print('No RETICOLO reference found. The meent-only checks above still apply.')
elif n_bad:
    print(f'RETICOLO agreement: {n_checked - n_bad}/{n_checked} points within '
          f'tolerance, {n_bad} over  ->  see section 4')
else:
    print(f'RETICOLO agreement: {n_checked}/{n_checked} points within '
          f'tolerance  ->  PASS')

In [ ]:
N_WORST = 10
worst.sort(reverse=True)
print(f"{'|dR|':>11s} {'pol':4s} {'method':11s} {'lambda':>9s}")
print('-' * 40)
for d, pol, method, lam in worst[:N_WORST]:
    print(f'{d:11.3e} {POLS[pol]:4s} {method:11s} {lam:8.0f}nm')

In [ ]:
from meent.on_torch.emsolver.convolution_matrix import (
    to_conv_mat_raster_discrete, to_conv_mat_raster_continuous)

BREAKDOWN_TOL = 1e-9

def stability_probe(pol, wl):
    mee = make_mee('discrete', pol, wl)
    A = to_conv_mat_raster_discrete(mee.ucell, fto[0], 0, device=mee.device,
                                    type_complex=mee.type_complex,
                                    enhanced_dfs=mee.enhanced_dfs,
                                    use_pinv=mee.use_pinv)
    B = to_conv_mat_raster_continuous(mee.ucell, fto[0], 0, device=mee.device,
                                      type_complex=mee.type_complex,
                                      use_pinv=mee.use_pinv)
    trace, dCB = [], None
    for t in (0.0, 0.5, 1.0 - 1e-12, 1.0):
        C = [A[i] + t * (B[i] - A[i]) for i in range(3)]
        if t != 1.0:
            dCB = max((C[i] - B[i]).abs().max().item() for i in range(3))
        r = mee.solve_for_conv(mee.wavelength, *C).res
        trace.append((t, float(r.de_ri.sum())))
    return trace, dCB

PROBE_THRESHOLD = 1e-4
probe_points = []
for d, pol, method, lam in worst:
    if d > PROBE_THRESHOLD and (pol, lam) not in probe_points:
        probe_points.append((pol, lam))
    if len(probe_points) >= 3:
        break
if probe_points:
    pol, lam = probe_points[0]
    step = (wavelength[1] - wavelength[0]) * 1e9
    if lam + step <= wavelength[-1] * 1e9:
        probe_points.append((pol, lam + step))
else:
    print(f'section 3 found nothing above {PROBE_THRESHOLD:g} - '
          f'probing one wavelength for reference')
    probe_points = [(1, wavelength[len(wavelength) // 2] * 1e9)]

print('t blends the discrete (t=0) and continuous (t=1) convolution matrices.')
print('Verdict compares t = 1 - 1e-12 against t = 1: those two matrices are')
print('numerically identical, so any difference in R is a breakdown.')
print()
for pol, lam in probe_points:
    trace, dCB = stability_probe(pol, lam * 1e-9)
    jump = abs(trace[-2][1] - trace[-1][1])
    verdict = 'BREAKDOWN' if jump > BREAKDOWN_TOL else 'stable'
    print(f'{POLS[pol]}  lambda={lam:.0f}nm')
    for t, r in trace:
        print(f'      t={t:<18.12g}  R={r:.9f}')
    print(f'      |C(1-1e-12) - C(1)| = {dCB:.2e}   ->  dR = {jump:.3e}   '
          f'->  {verdict}')
    print()

In [ ]:
styles = {'discrete': '--', 'continuous': ':', 'vector': '-.'}

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, squeeze=False)
for j, pol in enumerate(POLS):
    ret = load_rows(out_name('RETICOLO', pol))
    for i, (quantity, col) in enumerate((('R', 1), ('T', 2))):
        ax = axes[i, j]
        for method in METHODS:
            a = load_rows(out_name('Meent', pol, method))
            if a is None:
                continue
            ax.plot(a[:, 0], a[:, col], styles[method], label=f'meent {method}')
        if ret is not None:
            ax.plot(ret[:, 0], ret[:, col], 'k-', lw=2, label='RETICOLO')
        ax.set_ylabel(quantity)
        ax.grid(alpha=.3)
        ax.set_title(f'{POLS[pol]} - {quantity}', fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel('wavelength (nm)')
axes[0, 0].legend(fontsize=8)
fig.suptitle(f'anisotropic grating on slab - conical incidence, '
             f'theta = {np.degrees(theta):.0f} deg, phi = {np.degrees(phi):.0f} deg, '
             f'(nx, ny, nz) = ({n_x}, {n_y}, {n_z})', y=1.0)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True, squeeze=False)
for j, pol in enumerate(POLS):
    ax = axes[0, j]
    ret = load_rows(out_name('RETICOLO', pol))
    if ret is None:
        ax.set_title(f'{POLS[pol]} - no RETICOLO reference')
        continue
    for method in METHODS:
        a = load_rows(out_name('Meent', pol, method))
        if a is None:
            continue
        lam, dev = [], []
        for row in a:
            j = np.where(np.isclose(ret[:, 0], row[0], atol=1e-6))[0]
            if j.size == 0 or not (np.isfinite(row[1]) and np.isfinite(ret[j[0], 1])):
                continue
            lam.append(row[0])
            dev.append(abs(row[1] - ret[j[0], 1]))
        if not lam:
            continue
        ax.semilogy(lam, np.array(dev) + 1e-18, styles[method],
                    label=f'meent {method}')
    for tol in sorted(set(TOL_RETICOLO.values())):
        ax.axhline(tol, color='r', lw=.8, ls='-', alpha=.5)
    ax.set_xlabel('wavelength (nm)')
    ax.set_ylabel('|R_meent - R_RETICOLO|')
    ax.set_title(f'{POLS[pol]} - deviation from RETICOLO', fontsize=10)
    ax.grid(alpha=.3, which='both')
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()